In [1]:
from glob import glob
import pandas as pd
from tqdm import tqdm

from locations import extract_urls
from events import parse_mmd_taxonomy, extract_events
from timespan import parse_timespan

In [2]:
root_path = "../data/schede mappatura/"

skip = {"david_ruth_FEGB_E_00007": {"rows": 6, "cols": 1}}

In [3]:
chrono_schede = glob(f"{root_path}*/chronotop*")
chrono_schede

['../data/schede mappatura/bruenn_jehoshua_IS_S_00028/chronotopoi_bruenn_jehoshua_bozza.xlsx',
 '../data/schede mappatura/david_ruth_FEGB_E_00007/chronotopi_ruth_david_FEGB_E_00007.xlsx',
 '../data/schede mappatura/bruenn_ charlotte_IS_S_00027/chronotopi_charlotte_bruenn_IS_S_00027 (file revisionato).xlsx',
 '../data/schede mappatura/bruenn_ charlotte_IS_S_00027/chronotopoi_charlotte_bruenn_IS_S_00027 (bozza).xlsx',
 '../data/schede mappatura/stern_IS_S_00142/chronotopi_josef_stern_IS_S_00142.xlsx']

## A list of individual sources for experimentation

ignored in the oveall logic

In [4]:
current = "bruenn_jehoshua_IS_S_00028/chronotopoi_bruenn_jehoshua_bozza.xlsx"
# current = "david_ruth_FEGB_E_00007/chronotopi_ruth_david_FEGB_E_00007.xlsx"
current = "bruenn_ charlotte_IS_S_00027/chronotopi_charlotte_bruenn_IS_S_00027 (file revisionato).xlsx"
current = "stern_IS_S_00142/chronotopi_josef_stern_IS_S_00142.xlsx"

After identification of all sources

# Shortlist processable sources

In [5]:
overview = {
    "bruenn_jehoshua_IS_S_00028/chronotopoi_bruenn_jehoshua_bozza.xlsx": [
        "Foglio1",
    ],
    # 'david_ruth_FEGB_E_00007/chronotopi_ruth_david_FEGB_E_00007.xlsx': ['Ruth L.David',
    #  'Mutter',
    #  'Vater',
    #  'Isaak Oppenheimer',
    #  'Ernst',
    #  'Werner',
    #  'Hanna',
    #  'Michael',
    #  'Fedora',
    #  'Liese',
    #  'Herbert Aron David',
    #  'Alice Urbach'],
    "bruenn_ charlotte_IS_S_00027/chronotopi_charlotte_bruenn_IS_S_00027 (file revisionato).xlsx": [
        "Charlotte Bruenn geb. Hirsch",
        "Ludwig Hirsch",
        "Gutta Hirsch geb. Dingfelder",
        "Alice Hirsch (Estryn)",
        "Erich Hirsch",
        "Walter Hirsch",
        "Alfons Justin Hirsch",
        "Margot Hirsch",
    ],
    "stern_IS_S_00142/chronotopi_josef_stern_IS_S_00142.xlsx": [
        "Josef Stern",
        "Julius Stern",
        "Esther Stern",
        "Sonja Stern",
        "Großvater M David Kaminka",
        "Großmutter M Betty Kaminka geb.",
        "Großvater V Isaak Stern",
        "Großmutter V Auguste Stern geb ",
        "Mutter 2 Claire Stern",
    ],
}

## Put all data together

In [6]:
first = True
df_list = []
for k, v in overview.items():
    for s in v:
        name = k.split("/")[0]
        # print(k,s)
        if name in skip:
            df = pd.read_excel(
                root_path + k, sheet_name=s, skiprows=skip[name]["rows"]
            ).iloc[:, skip[name]["cols"] :]
        else:
            df = pd.read_excel(root_path + k, sheet_name=s)

        print(k, s, df.columns)
        df.columns = ["event", "location", "timespan", "notes", "links"]

        # merge columns 6+ to notes
        df["notes"] = df[["notes"] + list(df.columns[6:])].apply(
            lambda row: " ".join(row.dropna().astype(str)), axis=1
        )
        df = df.drop(df.columns[6:], axis=1)

        df["protagonist"] = name
        df["name"] = s
        df_list += [df]
df = pd.concat(df_list, axis=0)
df.fillna("", inplace=True)
df

bruenn_jehoshua_IS_S_00028/chronotopoi_bruenn_jehoshua_bozza.xlsx Foglio1 Index(['EVENT', 'LOCATION', 'TIMESPAN', 'NOTES', 'LINKS'], dtype='str')
bruenn_ charlotte_IS_S_00027/chronotopi_charlotte_bruenn_IS_S_00027 (file revisionato).xlsx Charlotte Bruenn geb. Hirsch Index(['Event', 'Location', 'Timespan', 'Notes', 'Links'], dtype='str')
bruenn_ charlotte_IS_S_00027/chronotopi_charlotte_bruenn_IS_S_00027 (file revisionato).xlsx Ludwig Hirsch Index(['Event', 'Location', 'Timespan', 'Notes', 'Links'], dtype='str')
bruenn_ charlotte_IS_S_00027/chronotopi_charlotte_bruenn_IS_S_00027 (file revisionato).xlsx Gutta Hirsch geb. Dingfelder Index(['Event', 'Location', 'Timespan', 'Notes', 'Links'], dtype='str')
bruenn_ charlotte_IS_S_00027/chronotopi_charlotte_bruenn_IS_S_00027 (file revisionato).xlsx Alice Hirsch (Estryn) Index(['Event', 'Location', 'Timespan', 'Notes', 'Links'], dtype='str')
bruenn_ charlotte_IS_S_00027/chronotopi_charlotte_bruenn_IS_S_00027 (file revisionato).xlsx Erich Hirsch

,event,location,timespan,notes,links,protagonist,name
0,Lebenslauf - Geburt / Alter Heimat,\t\nAllenstein (Ostpreußen)https://www.geoname...,19. Januar 1913,,,bruenn_jehoshua_IS_S_00028,Foglio1
1,Lebenslauf - Tod des Vaters,Allenstein,1923,24min01s. - 24min04s.,,bruenn_jehoshua_IS_S_00028,Foglio1
2,Lebenslauf - Schuljahren (Realschule),Allenstein,ca. 1927/1928,"Zuerst christliche Volksschule, dann (von der ...",,bruenn_jehoshua_IS_S_00028,Foglio1
3,Lebenslauf - Kaufmännische Lehre,Allenstein,1931,,,bruenn_jehoshua_IS_S_00028,Foglio1
4,Lebenslauf - Militärversuch,Allenstein,1931,Er hat 3 Jahren bei Frankenstein & Co (Warenha...,,bruenn_jehoshua_IS_S_00028,Foglio1
...,...,...,...,...,...,...,...
1,Deportation,Gießen https://www.wikidata.org/wiki/Q7904 Wal...,16.9.1942\n,,,stern_IS_S_00142,Mutter 2 Claire Stern
2,Transport (Abfahrt),Darmstadt https://www.wikidata.org/wiki/Q2973 ...,1942-09-30 00:00:00,,https://memoryoftreblinka.org/transports-to-tr...,stern_IS_S_00142,Mutter 2 Claire Stern
3,Transport (Ankunft),Treblinka https://www.wikidata.org/wiki/Q15201...,1942-10-02 00:00:00,,,stern_IS_S_00142,Mutter 2 Claire Stern
4,Mord,Treblinka https://www.wikidata.org/wiki/Q15201...,1942,Stolperstein Gießen,https://collections.yadvashem.org/en/names/135...,stern_IS_S_00142,Mutter 2 Claire Stern


# Locations

In [7]:
# locs = {n:l for l, n in df["location"].apply(lambda x: extract_urls(x)).to_list()}
locs = {}
for row in tqdm(df["location"]):
    # print(row)
    # print(extract_urls(row))
    for name, urls in extract_urls(row):
        urls["location"] = name
        locs[name] = urls
# print(locs)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 153/153 [00:00<00:00, 6140.53it/s]


## Update preexisting locations

In [8]:
rich = pd.read_excel("locations.xlsx")
rich = rich.set_index(["location"])
rich
for location, row in locs.items():
    if location not in rich.index:
        print(location)
        rich.loc[location] = locs[location]
rich
rich.to_excel("locations.xlsx")

Berling (Kolonnenstraße)
Palästina (über Marseille)
Ägypten/Irak/Iran

von Homburg (Saar) nach Nyons, Südfrankreich (bei Avignon)
Nyons Südfrankreich
Nyons, Südfrankreich
Israel (Ramatayim / Ramat HaSharon)
vermutlich Homburg (Saar)
von Homburg nach Südfrankreich (bei Avignon)
von Homburg (Saar) nach Südfrankreich (bei Avignon)
Südfrankreich?
aus dem Kibbuz ausgewiesen
Italien Deutschland Belgien Niederlande
Verkehrsmittel > Zug Klein Linden
Gießen  Walltorstraße 48 Darmstadt
aus Gießen nach Palästina
Von Berlin nach Theresienstadt


# Events

TODO: incomplete due to too much noise. Issues:

- use LL or Lebenslauf, currently extracted as one, but need to be two equivalent
- "alter heimant" instread of "alte heimat"
- "Transport", "Tod des Vaters",  are not label

In [9]:
event_taxonomy = parse_mmd_taxonomy("../model/tassonomia.mmd")
events = sorted(set(event_taxonomy.keys()), key=lambda x: -len(x))
len(events), events[:5] + ["..."] + events[-5:]

(340,
 ['Città/regioni tedesche, austriache, ceche',
  'SR – Spazi sociali / istituzioni',
  'Friedrichswerdersche Gymnasium',
  'Europa – altri paesi e luoghi',
  'Gymnasium zum Grauen Kloster',
  '...',
  'USA',
  'IPD',
  'Tod',
  'Zug',
  'SPD'])

In [10]:
df["events"] = df["event"].apply(lambda x: extract_events(x, events))
df

,event,location,timespan,notes,links,protagonist,name,events
0,Lebenslauf - Geburt / Alter Heimat,\t\nAllenstein (Ostpreußen)https://www.geoname...,19. Januar 1913,,,bruenn_jehoshua_IS_S_00028,Foglio1,"[Geburt, Lebenslauf - / Alter Heimat]"
1,Lebenslauf - Tod des Vaters,Allenstein,1923,24min01s. - 24min04s.,,bruenn_jehoshua_IS_S_00028,Foglio1,"[Tod, Lebenslauf - des Vaters]"
2,Lebenslauf - Schuljahren (Realschule),Allenstein,ca. 1927/1928,"Zuerst christliche Volksschule, dann (von der ...",,bruenn_jehoshua_IS_S_00028,Foglio1,"[Realschule, Lebenslauf - Schuljahren ()]"
3,Lebenslauf - Kaufmännische Lehre,Allenstein,1931,,,bruenn_jehoshua_IS_S_00028,Foglio1,[Lebenslauf - Kaufmännische Lehre]
4,Lebenslauf - Militärversuch,Allenstein,1931,Er hat 3 Jahren bei Frankenstein & Co (Warenha...,,bruenn_jehoshua_IS_S_00028,Foglio1,[Lebenslauf - Militärversuch]
...,...,...,...,...,...,...,...,...
1,Deportation,Gießen https://www.wikidata.org/wiki/Q7904 Wal...,16.9.1942\n,,,stern_IS_S_00142,Mutter 2 Claire Stern,[Deportation]
2,Transport (Abfahrt),Darmstadt https://www.wikidata.org/wiki/Q2973 ...,1942-09-30 00:00:00,,https://memoryoftreblinka.org/transports-to-tr...,stern_IS_S_00142,Mutter 2 Claire Stern,"[Abfahrt, Transport ()]"
3,Transport (Ankunft),Treblinka https://www.wikidata.org/wiki/Q15201...,1942-10-02 00:00:00,,,stern_IS_S_00142,Mutter 2 Claire Stern,"[Ankunft, Transport ()]"
4,Mord,Treblinka https://www.wikidata.org/wiki/Q15201...,1942,Stolperstein Gießen,https://collections.yadvashem.org/en/names/135...,stern_IS_S_00142,Mutter 2 Claire Stern,[Mord]


# Timespan

In [11]:
df[["time_start", "time_end"]] = (
    df["timespan"].apply(lambda ts: list(parse_timespan(ts).as_tuple())).tolist()
)
df

,event,location,timespan,notes,links,protagonist,name,events,time_start,time_end
0,Lebenslauf - Geburt / Alter Heimat,\t\nAllenstein (Ostpreußen)https://www.geoname...,19. Januar 1913,,,bruenn_jehoshua_IS_S_00028,Foglio1,"[Geburt, Lebenslauf - / Alter Heimat]",1913-01-19,1913-01-19
1,Lebenslauf - Tod des Vaters,Allenstein,1923,24min01s. - 24min04s.,,bruenn_jehoshua_IS_S_00028,Foglio1,"[Tod, Lebenslauf - des Vaters]",1923-01-01,1923-12-31
2,Lebenslauf - Schuljahren (Realschule),Allenstein,ca. 1927/1928,"Zuerst christliche Volksschule, dann (von der ...",,bruenn_jehoshua_IS_S_00028,Foglio1,"[Realschule, Lebenslauf - Schuljahren ()]",1927-01-01,1928-12-31
3,Lebenslauf - Kaufmännische Lehre,Allenstein,1931,,,bruenn_jehoshua_IS_S_00028,Foglio1,[Lebenslauf - Kaufmännische Lehre],1931-01-01,1931-12-31
4,Lebenslauf - Militärversuch,Allenstein,1931,Er hat 3 Jahren bei Frankenstein & Co (Warenha...,,bruenn_jehoshua_IS_S_00028,Foglio1,[Lebenslauf - Militärversuch],1931-01-01,1931-12-31
...,...,...,...,...,...,...,...,...,...,...
1,Deportation,Gießen https://www.wikidata.org/wiki/Q7904 Wal...,16.9.1942\n,,,stern_IS_S_00142,Mutter 2 Claire Stern,[Deportation],1942-09-16,1942-09-16
2,Transport (Abfahrt),Darmstadt https://www.wikidata.org/wiki/Q2973 ...,1942-09-30 00:00:00,,https://memoryoftreblinka.org/transports-to-tr...,stern_IS_S_00142,Mutter 2 Claire Stern,"[Abfahrt, Transport ()]",1942-09-30,1942-09-30
3,Transport (Ankunft),Treblinka https://www.wikidata.org/wiki/Q15201...,1942-10-02 00:00:00,,,stern_IS_S_00142,Mutter 2 Claire Stern,"[Ankunft, Transport ()]",1942-10-02,1942-10-02
4,Mord,Treblinka https://www.wikidata.org/wiki/Q15201...,1942,Stolperstein Gießen,https://collections.yadvashem.org/en/names/135...,stern_IS_S_00142,Mutter 2 Claire Stern,[Mord],1942-01-01,1942-12-31


# Notes

left unprocessed for now

In [12]:
set(df["notes"])

{'',
 ' ',
 ' („so eine Art Hachschara“)',
 ' Bruder nach Oslo',
 ' Scheda Lea - lebte unter falschem Namen „Alexandre Hilaire“ ',
 ' falscher Name „Marguerite Hilaire“ - Scheda LEA',
 ' starb schwer krank, nicht durch NS-Verfolgung  (Interview)',
 '(Fragebogen)',
 '(Fragebogen):',
 '(Gedenkblatt)',
 '(Interview Metz)',
 '1973: Jom Kippur Krieg ',
 '20 min 56',
 '22min 0s ',
 '24min01s. - 24min04s.',
 '26min 58s',
 '29min 04',
 '30min 57s',
 '31min 31s',
 '31min 58s',
 '32min 16s',
 '36min 51smein mittlerer Bruder weiß man bis heut nicht, er ist umgekommen, er war damals neunzehn Jahre in in Homburg + . Man hat ihn unter der Eisenbahn gefunden, man weiß bis heute nicht, ja. Das war diese Zeit, . Scheda LEA: 27.02.1936 mit den Eltern nach Nyons ausgewandert (https://gedenkbuch.saarbruecken.de/en/memorial_book/person_details/person-1182) Death 24 May 1935 Homburg, Saarpfalz\nSuicide',
 '39-40 CB: Fünfunddreißig nach der Abstimmung wie die Saar wurde deutsch und sind wir gleich',
 '39-40 

# Links

left unprocessed for now

In [13]:
set(df["links"])

{'',
 '\nhttps://collections.yadvashem.org/en/names/13219727 Esther-Stern-Platz Gießen  https://www.lotta-magazin.de/ausgabe/1/den-ermordeten-menschen-ein-gesicht-geben/ \n',
 '\nhttps://www.giessen.de/index.php?ModID=7&FID=2874.2967.1&object=tx%2C2874.2967.1 ',
 ' https://gedenkbuch.saarbruecken.de/en/memorial_book/person_details/person-1182',
 ' https://gedenkbuch.saarbruecken.de/gedenkbuch/personen_detailseite/person-3895',
 'Bahnof? https://www.wikidata.org/wiki/Q2915713',
 "Informazione che non compare nell'intervista, ma documentata nella scheda LEA: https://gedenkbuch.saarbruecken.de/en/memorial_book/person_details/person-1182",
 'Nell\'intervista C. dice "Mit 26 Jahren" (ergo nel 1949), ma in altre fonti risulta che si è sposata nel 1951 (Scheda LEA: https://gedenkbuch.saarbruecken.de/en/memorial_book/person_details/person-1182; © Institut für Deutsche Sprache, Mannheim 2013)',
 'http://www.ajpn.org/personne-Charlotte-Hirsch-10313.html https://gedenkbuch.saarbruecken.de/en/memo